In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:64"

import utils_cuda_tqdm as utils

import torch
from torch import nn
import torch.nn.functional as F


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_float32_matmul_precision('high')
torch.backends.cudnn.benchmark = True

print(torch.cuda.get_device_name())

# Loading the CIFAR10 dataset and create DataLoaders

In [ ]:
batch_size = 2048
train_iter, test_iter = utils.load_data_cifar10(batch_size)

# Defining the Model

In [ ]:

class MathisBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, num_experts: int, factor: int, downsample: bool = False):
        super().__init__()
        self.num_experts = num_experts
        reduced_channels = in_channels // factor

        # Learnable temperature parameter for the gating mechanism.
        self.temperature = nn.Parameter(torch.tensor(1.0))

        # Gating branch with dropout regularisation.
        self.expertBranch = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(in_channels, reduced_channels),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.2),
            nn.Linear(reduced_channels, num_experts)
            # Softmax is applied in forward() using self.temperature.
        )

        # Define expert modules: each expert is Conv -> BN -> ReLU
        self.expert_convs = nn.ModuleList([
            nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1)
            for _ in range(num_experts)
        ])
        self.expert_bns = nn.ModuleList([
            nn.BatchNorm2d(in_channels)
            for _ in range(num_experts)
        ])

        # Decide stride based on downsample flag.
        stride = 2 if downsample else 1

        # 1x1 convolution for channel projection, possibly with stride=2 to reduce spatial resolution.
        self.project = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=1,
            stride=stride
        )

        # Skip connection: if in_channels != out_channels or downsample, we use a 1x1 conv (stride=2 if needed).
        if downsample or in_channels != out_channels:
            self.skip = nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=1,
                stride=stride
            )
        else:
            self.skip = nn.Identity()

        # BatchNorm applied after merging the residual and expert outputs.
        self.bn_after_skip = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Compute gating logits and apply temperature-scaled softmax.
        gating_logits = self.expertBranch(x)
        gating = F.softmax(gating_logits / self.temperature, dim=-1)
        gating = gating.view(gating.size(0), self.num_experts, 1, 1, 1)

        # Pass input through each expert (Conv -> BN -> ReLU).
        expert_outputs = []
        for conv, bn in zip(self.expert_convs, self.expert_bns):
            expert_output = self.relu(bn(conv(x)))
            expert_outputs.append(expert_output)

        # Stack the expert outputs and combine using gating weights.
        expert_outputs = torch.stack(expert_outputs, dim=1)
        combined_expert_output = (expert_outputs * gating).sum(dim=1)

        # Project channels (and possibly downsample spatially).
        combined_expert_output = self.project(combined_expert_output)

        # Residual connection
        out = self.skip(x) + combined_expert_output
        out = self.bn_after_skip(out)
        out = self.relu(out)
        return out


class MathisNet(nn.Module):
    """
    MathisNet constructs a convolutional network by combining an initial stem,
    multiple MathisBlocks, and a final classifier head.
    """
    def __init__(self, conv_arch: list, num_outputs: int) -> None:
        super().__init__()
        # Initial stem: simple conv + BN + ReLU
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        )

        # Build the sequence of MathisBlocks from conv_arch
        in_channels = 64
        self.blocks = nn.ModuleList()
        for (out_channels, num_experts, factor, downsample) in conv_arch:
            self.blocks.append(MathisBlock(in_channels, out_channels, num_experts, factor, downsample))
            in_channels = out_channels

        # Classifier: global pooling + dropout + linear
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(in_channels, num_outputs)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.stem(x)
        for block in self.blocks:
            x = block(x)
        x = self.classifier(x)
        return x

# Initialise Model and Weights

In [ ]:
def init_weights(m):
    if isinstance(m, nn.Conv2d) or isinstance(m, nn.Linear):
        torch.nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
        if m.bias is not None:
            nn.init.constant_(m.bias, 0)

conv_arch = [
    (64,  4, 2, False),  # 32x32
    (128, 6, 4, True),   # 16x16
    (128, 6, 4, False),  # 16x16
    (256, 8, 4, True),   # 8x8
    (512, 8, 4, True),   # 4x4 → best high-level features
]
num_outputs = 10


model = MathisNet(conv_arch, num_outputs)
model.apply(init_weights)
model = model.to(device)
model = torch.compile(model)
print(model)

# Loss and Optimisation Algorithms

In [ ]:
loss = nn.CrossEntropyLoss(label_smoothing=0.1)
lr = 0.1
wd = 5e-4

optimiser = torch.optim.SGD(
    model.parameters(),
    lr=lr,
    momentum=0.9,
    weight_decay=wd,
    nesterov=True
)

# Training functions

In [ ]:
num_epochs = 200

scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimiser, max_lr=0.1, epochs=num_epochs, steps_per_epoch=len(train_iter)
)
utils.train(model, train_iter, test_iter, loss, num_epochs, optimiser, scheduler)

In [12]:
def test_model(model, test_loader):
    model.eval()
    with torch.no_grad():
        acc = utils.evaluate_accuracy(model, test_loader, device)
    print(f"✅ Test Accuracy: {acc * 100:.2f}%")
    return acc

print(test_model(model, train_iter))